In [1]:
from typing import Any
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import CIFAR10
from torchvision.models import resnet18
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms import ToTensor
import scan

Modify dataset and model for pretraining.

In [2]:
class CifarForScan(CIFAR10):
    def __getitem__(self, idx: int):
        img, _ = super().__getitem__(idx)
        return idx, img

In [3]:
class ResnetForScan(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self._resnet = resnet18(num_classes=10)
        self._extractor = create_feature_extractor(
            self._resnet,
            return_nodes=['avgpool']
        )
    
    def forward(self, img: torch.Tensor) -> torch.Tensor:
        return self._extractor(img)['avgpool'].view(-1, 512)

Forward pass

In [4]:
train_data = CifarForScan(
    root='C:/Users/chiwe/Data',
    train=True,
    transform=ToTensor()
)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
model = ResnetForScan().cuda()
optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

# instance discrimination memory bank
torch.manual_seed(0)
memory = torch.randn(len(train_data), 512)
memory = nn.functional.normalize(memory, p=2, dim=-1)

# example forward pass
idcs, imgs = next(iter(train_loader))
features = model(imgs.cuda())
from_mem = memory[idcs].cuda()
logits = features @ from_mem.T
loss = logits.softmax(dim=-1).diag().log().mean().neg()
optimizer.zero_grad()
loss.backward()
optimizer.step()